# 05 — Category Capital and Multi-Desk Analysis

The non-securitisation category DRC is the **sum of all bucket capitals**.

This notebook:
1. Reconciles against the committed golden expected outputs.
2. Shows a multi-desk breakdown using scoped context runs (one desk at a time).

*Regulatory refs*: Basel MAR22.20; US NPR § 210(b)(3)(iii)

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from examples.drc_nonsec_fixture import load_drc_nonsec_v2_fixture, run_fixture_workflow
from frtb_drc.demo_data import ALL_POSITIONS, DEMO_CONTEXT

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


fixture = load_drc_nonsec_v2_fixture()
positions = fixture.positions
context = fixture.context
expected = fixture.expected_outputs
display(
    Markdown(
        f"Loaded fixture `{expected['fixture_id']}`: "
        f"`{len(positions)}` positions, "
        f"profile `{context.profile_id}`, "
        f"as-of `{context.calculation_date}`."
    )
)

## Full capital assembly

In [ ]:
from frtb_drc.scaffold import calculate_drc_capital

result = calculate_drc_capital(positions, context=context)
cat = result.categories[0]

print(f"Input count  : {result.input_count}")
print(f"Input hash   : {result.input_hash}")
print(f"Profile hash : {result.profile_hash}")
print()
for b in cat.bucket_results:
    print(f"  {b.bucket_key:<22}  capital = {b.capital:>14,.2f}  hbr = {b.hbr.ratio:.4f}")
print(f"  {'Category total':<22}  capital = {cat.capital:>14,.2f}")
print(f"  {'Total DRC':<22}  capital = {result.total_drc:>14,.2f}")

## Golden reconciliation

In [ ]:
tolerance = 1e-9
checks = {
    "input_count"     : (result.input_count, expected["input_count"]),
    "input_hash"      : (result.input_hash, expected["input_hash"]),
    "profile_hash"    : (result.profile_hash, expected["profile_hash"]),
    "category_capital": (cat.capital, expected["category_capital"]),
    "total_drc"       : (result.total_drc, expected["total_drc"]),
}
rows = []
for name, (actual, golden) in checks.items():
    if isinstance(actual, float):
        ok = abs(actual - golden) < tolerance
        rows.append([name, f"{actual:,.4f}", f"{golden:,.4f}", "PASS" if ok else "FAIL"])
    else:
        ok = actual == golden
        rows.append([name, str(actual)[:40], str(golden)[:40], "PASS" if ok else "FAIL"])

display(markdown_table(["Check", "Actual", "Golden", "Status"], rows))

## Multi-desk breakdown

The portfolio spans three desks:
- **credit-desk**: CORPORATE positions
- **rates-desk**: NON\_US\_SOVEREIGN positions
- **structured-desk**: PSE\_GSE and DEFAULTED positions

Each desk can be run independently using a scoped context (`desk_id` filter).

In [ ]:
from frtb_drc.data_models import DrcCalculationContext
import matplotlib.pyplot as plt

desks = ["credit-desk", "rates-desk", "structured-desk"]
desk_results = {}
for desk in desks:
    desk_context = DrcCalculationContext(
        run_id=f"demo-{desk}",
        calculation_date=context.calculation_date,
        base_currency=context.base_currency,
        profile_id=context.profile_id,
        desk_id=desk,
    )
    desk_positions = [p for p in positions if p.desk_id == desk]
    desk_results[desk] = calculate_drc_capital(desk_positions, context=desk_context)

rows = []
for desk, res in desk_results.items():
    cat_d = res.categories[0]
    for b in cat_d.bucket_results:
        rows.append([desk, b.bucket_key, f"{b.capital:>14,.2f}", f"{b.hbr.ratio:.4f}"])
    rows.append([desk, "TOTAL", f"{res.total_drc:>14,.2f}", ""])

display(markdown_table(["Desk", "Bucket", "Capital", "HBR"], rows))

# Verify desk totals sum to portfolio total
desk_total = sum(r.total_drc for r in desk_results.values())
portfolio_total = result.total_drc
print(f"\nSum of desk capitals : {desk_total:,.2f}")
print(f"Portfolio total      : {portfolio_total:,.2f}")
print(f"Additive             : {abs(desk_total - portfolio_total) < 1e-9}")

## Capital by desk and bucket

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

all_buckets = ["CORPORATE", "NON_US_SOVEREIGN", "PSE_GSE", "DEFAULTED"]
colours = {"CORPORATE": "#2563EB", "NON_US_SOVEREIGN": "#059669",
           "PSE_GSE": "#D97706", "DEFAULTED": "#DC2626"}
desk_labels = desks

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(desk_labels))
bottom = np.zeros(len(desk_labels))
for bucket in all_buckets:
    heights = []
    for desk in desk_labels:
        cat_d = desk_results[desk].categories[0]
        bkt_cap = next(
            (b.capital / 1e6 for b in cat_d.bucket_results if b.bucket_key == bucket), 0.0
        )
        heights.append(bkt_cap)
    ax.bar(x, heights, bottom=bottom, label=bucket, color=colours[bucket], alpha=0.85)
    bottom += np.array(heights)

ax.set_xticks(x)
ax.set_xticklabels(desk_labels, fontsize=10)
ax.set_ylabel("DRC capital (USD M)")
ax.set_title("DRC capital by desk and bucket")
ax.legend(fontsize=8, loc="upper right")
fig.tight_layout()
plt.show()

## Audit trail

In [ ]:
from frtb_drc import result_json
import hashlib

rj = result_json(result)
sha = hashlib.sha256(rj.encode()).hexdigest()
print(f"result_json SHA-256: {sha}")
print(f"Golden SHA-256:      {expected['result_json_sha256']}")
print(f"Match: {sha == expected['result_json_sha256']}")
print()
print(f"Citations used ({len(result.citations)}):")
for cit in result.citations:
    print(f"  {cit}")